# Step One: Supervised Baseline
### for Assignment 2: SSL
**By Tahiyat Chy**

This initial model is trained using only the classification objective and will act as the baseline model.

## Initial Setup

In [9]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA available: True


In [10]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [11]:
# Path set to assignment directory

PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169/assignment2')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test


In [12]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation
Define data augmentations. Split the training images into training and validation set.

In [13]:
import torchvision.transforms as T

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_CLASSES = 16

# Data augmentation
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandAugment(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1)
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_ROOT)

# Split into train and val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Implement custom wrapper to apply different transforms to train and val subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

Total training images: 1920
Total validation images: 480


## Model Architecture
Vision Transformer (ViT-B/16) initialized with weights=None.

In [14]:
from torchvision.models import vit_b_16

def get_baseline_model():
    # Initialize model
    model = vit_b_16(weights=None)
    # Adjust head for 16 classes
    model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    return model.to(device)

model = get_baseline_model()
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model parameters: 85,810,960


## Training Loop
Using CrossEntropyLoss and AdamW optimizer.

In [15]:
def train_model(model, train_loader, val_loader, epochs=20, lr=3e-4):
    # Label smoothing to prevent overconfidence
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    # AdamW for better weight decay regularization in Transformers compared to Adam
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)

        # Validation
        model.eval()
        corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_acc = corrects.double() / len(val_loader.dataset)
        history['val_acc'].append(epoch_acc.item())

        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Acc: {epoch_acc:.4f}')

        scheduler.step(epoch_acc)

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    print(f'\nTraining complete. Best Validation Accuracy: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

# Execute training with 20 epochs initially (using lower lr for ViT)
baseline_model, history = train_model(model, train_loader, val_loader, epochs=20, lr=3e-4)

Epoch 1/20 | Train Loss: 3.1025 | Val Acc: 0.1062
Epoch 2/20 | Train Loss: 2.6892 | Val Acc: 0.1333
Epoch 3/20 | Train Loss: 2.6249 | Val Acc: 0.1750
Epoch 4/20 | Train Loss: 2.5878 | Val Acc: 0.2312
Epoch 5/20 | Train Loss: 2.4644 | Val Acc: 0.2188
Epoch 6/20 | Train Loss: 2.4210 | Val Acc: 0.2521
Epoch 7/20 | Train Loss: 2.3839 | Val Acc: 0.2167
Epoch 8/20 | Train Loss: 2.4180 | Val Acc: 0.2292
Epoch 9/20 | Train Loss: 2.3261 | Val Acc: 0.2854
Epoch 10/20 | Train Loss: 2.3233 | Val Acc: 0.3021
Epoch 11/20 | Train Loss: 2.2938 | Val Acc: 0.2750
Epoch 12/20 | Train Loss: 2.2852 | Val Acc: 0.2833
Epoch 13/20 | Train Loss: 2.2838 | Val Acc: 0.2250
Epoch 14/20 | Train Loss: 2.2104 | Val Acc: 0.3458
Epoch 15/20 | Train Loss: 2.2146 | Val Acc: 0.3208
Epoch 16/20 | Train Loss: 2.1513 | Val Acc: 0.3625
Epoch 17/20 | Train Loss: 2.1483 | Val Acc: 0.3521
Epoch 18/20 | Train Loss: 2.1090 | Val Acc: 0.3187
Epoch 19/20 | Train Loss: 2.1258 | Val Acc: 0.3771
Epoch 20/20 | Train Loss: 2.0936 | Val A